<a href="https://colab.research.google.com/github/RsSubhamMohanty/The-Reasoners-Educational-Content-Generator-AI-Agent-Development-Project-Dual-Track-Version-/blob/main/Educational_Agent_Week8.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install groq

In [ ]:
from groq import Groq

In [ ]:
client = Groq(api_key="")

In [ ]:
response = client.chat.completions.create(
    model="llama-3.1-8b-instant",
    messages=[{"role":"user","content":"Explain Artificial Intelligence in 3 simple lines"}]
)

print(response.choices[0].message.content)

In [ ]:
def generate_quiz(study_text):

    prompt = f"""
    You are an AI Study Assistant.

    From the study material below create 5 multiple choice questions.

    Study Material:
    {study_text}

    Format:

    Question
    A)
    B)
    C)
    D)
    Correct Answer
    """

    response = client.chat.completions.create(
        model="llama-3.1-8b-instant",
        messages=[{"role":"user","content":prompt}]
    )

    return response.choices[0].message.content

In [ ]:
study_notes = """
Machine Learning is a branch of Artificial Intelligence that allows computers
to learn patterns from data and improve performance without being explicitly programmed.
"""

In [ ]:
quiz = generate_quiz(study_notes)

print(quiz)

In [ ]:
!pip install PyPDF2

In [ ]:
import PyPDF2

In [ ]:
from google.colab import files

uploaded = files.upload()

In [ ]:
pdf_text = ""

for file_name in uploaded.keys():
    pdf_reader = PyPDF2.PdfReader(file_name)

    for page in pdf_reader.pages:
        pdf_text += page.extract_text()

print("PDF text extracted successfully")

In [ ]:
print(pdf_text[:1000])

In [ ]:
quiz = generate_quiz(pdf_text)

print(quiz)

In [ ]:
!pip install chromadb
!pip install sentence-transformers
!pip install langchain-community
!pip install langchain-text-splitters

In [ ]:
!pip uninstall -y typer click
!pip install typer==0.9.0 click==8.1.7

In [ ]:
from langchain_text_splitters import CharacterTextSplitter
from langchain_community.vectorstores import Chroma
from langchain_community.embeddings import HuggingFaceEmbeddings

In [ ]:
text_splitter = CharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=200
)

chunks = text_splitter.split_text(pdf_text)

print("Total chunks:", len(chunks))

In [ ]:
embeddings = HuggingFaceEmbeddings()

In [ ]:
vector_db = Chroma.from_texts(
    chunks,
    embeddings
)

print("Vector database created")

In [ ]:
query = "Create quiz questions from this study material"

docs = vector_db.similarity_search(query)

context = docs[0].page_content

print(context[:500])

In [ ]:
prompt = f"""
Create 5 multiple choice questions from the following study material.

{context}
"""

response = client.chat.completions.create(
    model="llama-3.1-8b-instant",
    messages=[{"role":"user","content":prompt}]
)

print(response.choices[0].message.content)

In [ ]:
!pip install langchain
!pip install langchain-community
!pip install duckduckgo-search

In [ ]:
def generate_summary(context):

    prompt = f"""
    Summarize the following study material in clear bullet points.

    Material:
    {context}
    """

    response = client.chat.completions.create(
        model="llama-3.1-8b-instant",
        messages=[{"role": "user", "content": prompt}]
    )

    return response.choices[0].message.content

In [ ]:
def generate_flashcards(context):

    prompt = f"""
    Create 5 study flashcards from this material.

    Format:
    Question:
    Answer:

    Material:
    {context}
    """

    response = client.chat.completions.create(
        model="llama-3.1-8b-instant",
        messages=[{"role":"user","content":prompt}]
    )

    return response.choices[0].message.content

In [ ]:
def generate_quiz(context):

    prompt = f"""
    Create 5 multiple choice questions.

    Material:
    {context}
    """

    response = client.chat.completions.create(
        model="llama-3.1-8b-instant",
        messages=[{"role":"user","content":prompt}]
    )

    return response.choices[0].message.content

In [ ]:
query = "study material"

docs = vector_db.similarity_search(query)

context = docs[0].page_content

In [ ]:
result = generate_flashcards(context)

print(result)

In [ ]:
!pip install streamlit
!pip install pyngrok

In [ ]:
pip install fpdf

In [ ]:
from google.colab import files
files.upload()


In [ ]:
import os
os.listdir()


In [ ]:
!mv "Home.png.png" home.png

In [ ]:
!pip install gtts

In [ ]:
!pip install youtube-search-python matplotlib

In [ ]:
!rm study.db

In [ ]:
%%writefile app.py

import streamlit as st
import PyPDF2
from groq import Groq
import sqlite3
from datetime import datetime
import matplotlib.pyplot as plt
import plotly.graph_objects as go
import pandas as pd
from datetime import datetime

# Install gtts if not already installed (for Streamlit app environment)
import subprocess
try:
    import gtts
except ImportError:
    subprocess.check_call(["pip", "install", "gtts"])

from gtts import gTTS

# ===== PAGE CONFIG =====
st.set_page_config(layout="wide")

# ===== UI ENHANCEMENT (ONLY ADDED) =====
st.markdown("""
<style>

/* 🌈 GOOGLE FONTS */
@import url('https://fonts.googleapis.com/css2?family=Poppins:wght@400;600;700&family=Roboto:wght@300;400&display=swap');

html, body, [class*="css"] {
    font-family: 'Roboto', sans-serif;
}

/* 🌿 BACKGROUND */
.stApp {
    background: linear-gradient(135deg, #6a11cb, #ff6ec4, #ff9a44, #00c9a7);
    background-size: 300% 300%;
    animation: gradientBG 10s ease infinite;
}

@keyframes gradientBG {
    0% {background-position: 0% 50%;}
    50% {background-position: 100% 50%;}
    100% {background-position: 0% 50%;}
}

/* 🚀 HEADER */
.header {
    background: linear-gradient(90deg, #ff9a44, #ff6ec4, #6a11cb);
    padding: 25px;
    border-radius: 20px;
    text-align: center;
    color: white;
    box-shadow: 0 10px 25px rgba(0,0,0,0.2);
}

.header h1 {
    font-family: 'Poppins', sans-serif;
    font-weight: 700;
}

.header p {
    color: #fff3b0;
}

/* 📦 CARD */
.card {
    background: rgba(255,255,255,0.1);
    padding: 20px;
    border-radius: 20px;
    backdrop-filter: blur(10px);
    box-shadow: 0 8px 25px rgba(0,0,0,0.2);
    transition: 0.3s;
}

.card:hover {
    transform: scale(1.05);
}

/* 🎯 BUTTON */
.stButton>button {
    background: linear-gradient(135deg, #8b5cf6, #6366f1);
    color: white !important;
    border-radius: 12px;
    border: none;
    padding: 10px;
    transition: 0.3s;
}

.stButton>button:hover {
    transform: scale(1.05);
    box-shadow: 0 0 15px rgba(139,92,246,0.6);
}

/* 📂 UPLOAD BOX */
.upload-box {
    background: rgba(0,0,0,0.6);
    padding: 20px;
    border-radius: 15px;
}

/* 🔥 LABEL FIX */
label {
    color: white !important;
    font-weight: 500;
}

/* ================= FILE UPLOADER FIX ================= */
div[data-testid="stFileUploader"] {
    background: #1e293b !important;
    border-radius: 12px;
    padding: 10px;
}

div[data-testid="stFileUploaderDropzone"] {
    background: #1e293b !important;
    border: 2px dashed #6366f1 !important;
}

div[data-testid="stFileUploader"] * {
    color: white !important;
}

div[data-testid="stFileUploaderFile"] {
    background: #0f172a !important;
    color: white !important;
    border-radius: 8px;
}

/* ================= SELECTBOX FIX ================= */
div[data-baseweb="select"] > div {
    background: #1e293b !important;
    color: white !important;
    border-radius: 10px !important;
}

div[data-baseweb="select"] span {
    color: white !important;
}

ul[role="listbox"] {
    background: #1e293b !important;
    color: white !important;
    border-radius: 10px;
}

/* ================= INPUT FIX ================= */
div[data-baseweb="input"] {
    background: #1e293b !important;
    border-radius: 10px;
}

div[data-baseweb="input"] input {
    color: white !important;
}

/* Remove inner white */
div[data-baseweb="base-input"] {
    background: transparent !important;
}

/* 🔥 Z-INDEX FIX */
div[data-baseweb="select"] {
    z-index: 999 !important;
}

/* 🔥 GLOBAL TEXT FIX */
p, span {
    color: white !important;
}

/* ================= GLASSMORPHISM UPGRADE ================= */

/* 🔥 Replace dark boxes with soft gradient glass */
div[data-baseweb="input"],
div[data-baseweb="select"] > div,
div[data-testid="stFileUploader"],
div[data-testid="stFileUploaderDropzone"] {
    background: linear-gradient(135deg, rgba(255,255,255,0.15), rgba(0,0,0,0.2)) !important;
    backdrop-filter: blur(15px) !important;
    -webkit-backdrop-filter: blur(15px) !important;
    border: 1px solid rgba(255,255,255,0.2) !important;
    box-shadow: 0 4px 20px rgba(0,0,0,0.2);
}

/* 🔥 File uploaded box */
div[data-testid="stFileUploaderFile"] {
    background: linear-gradient(135deg, rgba(255,255,255,0.1), rgba(0,0,0,0.3)) !important;
    backdrop-filter: blur(10px);
}

/* 🔥 Dropdown menu */
ul[role="listbox"] {
    background: linear-gradient(135deg, rgba(30,41,59,0.95), rgba(0,0,0,0.8)) !important;
    backdrop-filter: blur(10px);
}

/* 🔥 Smooth glow effect */
div[data-baseweb="input"]:hover,
div[data-baseweb="select"] > div:hover {
    box-shadow: 0 0 15px rgba(255,255,255,0.2);
}

/* 🔥 Make upload box match theme */
.upload-box {
    background: linear-gradient(135deg, rgba(255,255,255,0.12), rgba(0,0,0,0.25)) !important;
    backdrop-filter: blur(15px);
}

/* 🔥 Slight card glow */
.card {
    box-shadow: 0 10px 30px rgba(255,255,255,0.1);
}
/* ===== FIX TEXT INSIDE WHITE SCHEDULE BOX ===== */

/* Target only the schedule cards (white boxes) */
div[data-testid="stHorizontalBlock"] div {
    color: black !important;
}

/* Make all text inside black */
div[data-testid="stHorizontalBlock"] * {
    color: black !important;
}
/* ================= STREAMLIT DATA EDITOR FIX ================= */

/* Main grid background */
.stDataFrame, .stDataEditor {
    background-color: white !important;
}

/* Glide Data Grid container */
div[data-testid="stDataEditor"] {
    background-color: white !important;
}

/* Actual table grid */
div[data-testid="stDataEditor"] div[role="grid"] {
    background-color: white !important;
}

/* Header */
div[data-testid="stDataEditor"] div[role="columnheader"] {
    background-color: #f1f5f9 !important;
    color: black !important;
}

/* Cells */
div[data-testid="stDataEditor"] div[role="gridcell"] {
    background-color: white !important;
    color: black !important;
}

/* Rows */
div[data-testid="stDataEditor"] div[role="row"] {
    background-color: white !important;
}

/* Force all text black */
div[data-testid="stDataEditor"] * {
    color: black !important;
}

/* Checkbox */
div[data-testid="stDataEditor"] input {
    accent-color: #6366f1;
}
/* ================= STUDY PLANNER HEADER BOX ================= */

.planner-header {
    background: linear-gradient(135deg, #ff6ec4, #ff9a44);
    padding: 18px 24px;
    border-radius: 16px;
    font-size: 26px;
    font-weight: 600;
    color: white;
    box-shadow: 0 6px 20px rgba(0,0,0,0.15);
    margin-bottom: 20px;
}
/* ================= COMMON SECTION HEADER ================= */

.section-header {
    background: linear-gradient(135deg, #ff6ec4, #ff9a44);
    padding: 18px 24px;
    border-radius: 16px;
    font-size: 26px;
    font-weight: 600;
    color: white;
    box-shadow: 0 6px 20px rgba(0,0,0,0.15);
    margin-bottom: 20px;
}
/* ================= RESULT BLUR BOX ================= */

.result-box {
    background: linear-gradient(
        135deg,
        rgba(255,255,255,0.25),
        rgba(255,255,255,0.1)
    );
    backdrop-filter: blur(15px);
    -webkit-backdrop-filter: blur(15px);

    padding: 25px;
    border-radius: 20px;

    border: 1px solid rgba(255,255,255,0.3);

    box-shadow: 0 8px 30px rgba(0,0,0,0.2);

    margin-top: 20px;
}

/* 🔥 TEXT INSIDE RESULT */
.result-box * {
    color: white !important;
}

</style>
""", unsafe_allow_html=True)

# ===== HEADER =====
st.markdown("""
<div class="header">
    <h1>🚀 AI Study Assistant</h1>
    <p>Stay consistent, success will follow 💡</p>
</div>
""", unsafe_allow_html=True)

# ===== GROQ =====
client = Groq(api_key="")

# ===== DATABASE =====
conn = sqlite3.connect("study.db", check_same_thread=False)
c = conn.cursor()

c.execute("""
CREATE TABLE IF NOT EXISTS history (
    id INTEGER PRIMARY KEY AUTOINCREMENT,
    type TEXT,
    topic TEXT,
    date TEXT
)
""")

c.execute("""
CREATE TABLE IF NOT EXISTS schedule (
    id INTEGER PRIMARY KEY AUTOINCREMENT,
    topic TEXT,
    study_date TEXT,
    study_time TEXT,
    task_type TEXT,
    status TEXT
)
""")

conn.commit()

# ===== SESSION =====
for key in ["quiz_result","flashcard_result","summary_result","audio_result","topics","selected_topic","active_tab"]:
    if key not in st.session_state:
        st.session_state[key] = "" if "result" in key else []

if "completed_topics" not in st.session_state:
    st.session_state.completed_topics = set()

if "action_triggered" not in st.session_state:
    st.session_state.action_triggered = None

if st.session_state.active_tab == []:
    st.session_state.active_tab = "quiz"

# ===== FUNCTIONS =====
def extract_text(file):
    try:
        reader = PyPDF2.PdfReader(file)
        text = ""
        for page in reader.pages:
            if page.extract_text():
                text += page.extract_text()
        if not text.strip():
            st.warning("⚠️ No readable text found in PDF")
        return text
    except Exception as e:
        st.error("❌ Error reading PDF")
        return ""

def ask_llm(prompt):
    try:
        response = client.chat.completions.create(
            model="llama-3.1-8b-instant",
            messages=[{"role":"user","content":prompt}]
        )
        return response.choices[0].message.content
    except Exception as e:
        st.error("⚠️ AI service failed. Try again.")
        return ""

def detect_topics(text):
    prompt = f"""
    Extract ONLY academic subject topics from the content.

    VERY IMPORTANT RULES:
    - Ignore metadata like: Credit, Lecture, Hrs/Week, Marks, Examination, Internal, External
    - Ignore headings, tables, syllabus structure
    - Ignore numbers, units, and formatting text
    - Only include REAL STUDY TOPICS (like Java Servlets, JDBC, JSP, MVC Architecture)
    - Each topic must be 2–5 words only
    - No single-word generic terms like "Credit", "Lecture"
    - No duplicates
    - Return only comma-separated topics

    Content:
    {text[:3000]}
    """

    topics = ask_llm(prompt)

    topic_list = [t.strip() for t in topics.split(",") if t.strip()]

    # ✅ Remove unwanted words manually (extra safety)
    unwanted = ["credit", "lecture", "marks", "hrs", "week", "internal", "external", "examination"]

    clean_topics = []
    for t in topic_list:
        if not any(word in t.lower() for word in unwanted):
            if 2 <= len(t.split()) <= 5:
                clean_topics.append(t)

    # ✅ remove duplicates
    clean_topics = list(dict.fromkeys(clean_topics))

    return clean_topics

def get_youtube_videos(topic):
    return [f"https://www.youtube.com/results?search_query={topic}+education"]

# ===== FILE  =====
st.markdown('<div class="card">', unsafe_allow_html=True)

st.markdown("### 📂 Upload PDFs")

# 🔳 Dark Upload Box
st.markdown('<div class="upload-box">', unsafe_allow_html=True)

uploaded_files = st.file_uploader(
    "Upload PDF",
    type=["pdf"],
    accept_multiple_files=True
)

st.markdown('</div>', unsafe_allow_html=True)

# 🔢 Number Input
num_questions = st.number_input("🔢 Number of Questions", 1, 20, 5)

# ===== LOGIC (UNCHANGED) =====
if "pdf_files" not in st.session_state:
    st.session_state.pdf_files = {}

if uploaded_files:
    for file in uploaded_files:
        st.session_state.pdf_files[file.name] = file

if st.session_state.pdf_files:


    st.markdown("<br>", unsafe_allow_html=True)

    selected_pdf = st.selectbox(
        "📚 Select Study Material",
        list(st.session_state.pdf_files.keys())
    )

    current_file = st.session_state.pdf_files[selected_pdf]

    st.markdown("<br>", unsafe_allow_html=True)

    # 🔥 Extract Topics Button
    if st.button("📊 Extract Topics", key="extract_topics_top_btn"):
      st.session_state.topics = detect_topics(extract_text(current_file))
      st.session_state.selected_topic = None
      st.rerun()

    st.markdown("<br>", unsafe_allow_html=True)

    # 🔥 SHOW TOPIC UI ONLY WHEN TOPICS EXIST
    if st.session_state.get("topics"):

      topics = st.session_state.topics

      # set default
      if not st.session_state.get("selected_topic"):
          st.session_state.selected_topic = topics[0]

      selected_topic = st.selectbox(
          "📘 Select Topic",
          topics,
          index=topics.index(st.session_state.selected_topic)
      )

      # 🔥 update state properly
      st.session_state.selected_topic = selected_topic

      st.markdown(f"""
      <div class="card" style="background:linear-gradient(135deg,#00c9a7,#6a11cb); color:white;">
          Selected Topic: <b>{selected_topic}</b>
      </div>
      """, unsafe_allow_html=True)

    st.markdown("<br>", unsafe_allow_html=True)

    if "last_selected_pdf" not in st.session_state:
        st.session_state.last_selected_pdf = None

    if selected_pdf != st.session_state.last_selected_pdf:
        st.session_state.topics = []
        st.session_state.completed_topics = set()
        st.session_state.last_selected_pdf = selected_pdf

st.markdown('</div>', unsafe_allow_html=True)

# ===== BUTTONS =====
st.markdown("## 🎯 Choose Action")

col1, col2, col3, col4 = st.columns(4)

def feature_card(title, desc, key, gradient):

    st.markdown(f"""
    <div class="card" style="background:{gradient}; text-align:center;">
        <h3 style="font-family:Poppins; color:white;">{title}</h3>
        <p style="color:#f1f5f9;">{desc}</p>
    </div>
    """, unsafe_allow_html=True)

    col1, col2, col3 = st.columns([1,2,1])

    with col2:
        if st.button("Open", key=key, use_container_width=True):
            st.session_state.action_triggered = key


with col1:
    feature_card("🧠 Quiz", "Generate MCQs instantly", "quiz",
                 "linear-gradient(135deg,#ff6ec4,#7873f5)")

with col2:
    feature_card("📘 Flashcards", "Quick revision mode", "flashcard",
                 "linear-gradient(135deg,#4facfe,#8b5cf6)")

with col3:
    feature_card("📄 Summary", "Short & clear notes", "summary",
                 "linear-gradient(135deg,#00c9a7,#6a11cb)")

with col4:
    feature_card("🎧 Audio", "Listen & learn", "audio",
                 "linear-gradient(135deg,#6a11cb,#4facfe)")

# ===== MAIN =====
if st.session_state.pdf_files:

    context = extract_text(current_file)

    if not context.strip():
        st.error(" No content extracted from PDF")
        st.stop()

    if not st.session_state.topics:
        st.info(" Click 'Extract Topics' to begin")
        st.stop()

    topics = st.session_state.topics

    if not st.session_state.selected_topic:
        st.session_state.selected_topic = topics[0]

    # ===== FIXED QUIZ =====
    if st.session_state.action_triggered == "quiz":
        st.session_state.active_tab = "quiz"
        st.session_state.quiz_result = ""

        with st.spinner("Generating Quiz..."):
            st.session_state.quiz_result = ask_llm(f"""
            Create {num_questions} MCQs from {selected_topic}:
            {context[:2000]}
            """)

        c.execute("INSERT INTO history VALUES (NULL,?,?,?)",
                  ("Quiz", selected_topic, str(datetime.now())))
        conn.commit()

        st.session_state.action_triggered = None

    if st.session_state.action_triggered == "flashcard":
        st.session_state.active_tab = "flashcard"
        st.session_state.flashcard_result = ""

        with st.spinner("Generating Flashcards..."):
            st.session_state.flashcard_result = ask_llm(f"""
            Create {num_questions} flashcards from {selected_topic}:
            {context[:2000]}
            """)

        c.execute("INSERT INTO history VALUES (NULL,?,?,?)",
                  ("Flashcard", selected_topic, str(datetime.now())))
        conn.commit()

        st.session_state.action_triggered = None

    if st.session_state.action_triggered == "summary":
        st.session_state.active_tab = "summary"
        st.session_state.summary_result = ""

        with st.spinner("Generating Summary..."):
            st.session_state.summary_result = ask_llm(f"""
            Summarize {selected_topic}:
            {context[:2000]}
            """)

        c.execute("INSERT INTO history VALUES (NULL,?,?,?)",
                  ("Summary", selected_topic, str(datetime.now())))
        conn.commit()

        st.session_state.action_triggered = None

    if st.session_state.action_triggered == "audio":
        st.session_state.active_tab = "audio"
        st.session_state.audio_result = ""

        with st.spinner("Generating Audio Explanation..."):
            st.session_state.audio_result = ask_llm(f"""
            Explain {selected_topic} simply:
            {context[:2000]}
            """)


        if st.session_state.audio_result:
            tts = gTTS(st.session_state.audio_result)
            tts.save("audio.mp3")
            st.audio("audio.mp3")

        st.session_state.action_triggered = None

    # ===== DISPLAY =====
    content = ""

    if st.session_state.active_tab == "quiz" and st.session_state.quiz_result:
        st.markdown("## 📝 Quiz")
        content = st.session_state.quiz_result

    elif st.session_state.active_tab == "flashcard" and st.session_state.flashcard_result:
        st.markdown("## 🧠 Flashcards")
        content = st.session_state.flashcard_result

    elif st.session_state.active_tab == "summary" and st.session_state.summary_result:
        st.markdown("## 📌 Summary")
        content = st.session_state.summary_result

    elif st.session_state.active_tab == "audio" and st.session_state.audio_result:
        st.markdown("## 🎧 Audio Explanation")
        content = st.session_state.audio_result

    if content:
        st.markdown(f"""
        <div class="result-box">
            {content.replace('\n', '<br>')}
        </div>
        """, unsafe_allow_html=True)

        st.download_button(
        label="⬇️ Download Result",
        data=content,
        file_name=f"{st.session_state.active_tab}_{selected_topic}.txt",
        mime="text/plain"
    )

    if st.session_state.active_tab == "flashcard":
        st.markdown("### 🔁 Review Again?")
        if st.button("Revise Flashcards"):
            st.session_state.flashcard_result = ""

    if st.session_state.active_tab == "audio":
        st.success("🎧 Audio ready! Listen below")


    # ===== STUDY PLANNER  =====
    st.markdown("<br><br>", unsafe_allow_html=True)
    st.markdown("""
    <div class="planner-header">
        📅 Study Planner
    </div>
    """, unsafe_allow_html=True)

    col_left, col_right = st.columns([1, 1])

    # ================= LEFT SIDE (FORM) =================
    with col_left:
        st.markdown('<div class="clean-card">', unsafe_allow_html=True)

        st.markdown("### ✏️ Plan Your Study")

        planner_topic = st.selectbox("📚 Select Topic", topics)

        colA, colB = st.columns(2)

        with colA:
            study_date = st.date_input("📅 Date")

        with colB:
            study_time = st.time_input("⏱ Time")

        task_type = st.selectbox(
            "🎯 Task Type",
            ["Read", "Practice", "Revise", "Quiz"]
        )

        st.markdown("<br>", unsafe_allow_html=True)

        if st.button("➕ Add Plan", use_container_width=True):

            c.execute(
                "INSERT INTO schedule (topic, study_date, study_time, task_type, status) VALUES (?, ?, ?, ?, ?)",
                (planner_topic, str(study_date), str(study_time), task_type, "Pending")
            )
            conn.commit()

            st.success(f"Added '{planner_topic}' to your study plan")

        st.markdown('</div>', unsafe_allow_html=True)


    # ================= RIGHT SIDE (SCHEDULE) =================
    with col_right:
        st.markdown('<div class="clean-card">', unsafe_allow_html=True)

        st.markdown("### 📌 Your Schedule")

        plans = c.execute(
            "SELECT id, topic, study_date, study_time, task_type, status FROM schedule"
        ).fetchall()

        if plans:
            for plan in plans:

                col1, col2, col3 = st.columns([4,1,1])

                # 🎨 STATUS COLOR
                status_color = "green" if plan[5] == "Done" else "orange"

                with col1:
                    st.markdown(f"""
                    <div style="
                        background:#f8fafc;
                        padding:12px;
                        border-radius:10px;
                        margin-bottom:8px;
                        border:1px solid #e2e8f0;
                    ">
                        <div style="font-weight:600;">{plan[1]}</div>
                        <div style="font-size:13px; color:#64748b;">
                            {plan[2]} • {plan[3]} • {plan[4]}
                        </div>
                        <div style="font-size:12px; margin-top:4px;">
                            Status: <span style="color:{status_color}; font-weight:600;">
                            {plan[5]}
                            </span>
                        </div>
                    </div>
                    """, unsafe_allow_html=True)

                # ✅ MARK DONE BUTTON
                with col2:
                    if plan[5] != "Done":
                        if st.button("✔", key=f"done_{plan[0]}", use_container_width=True):
                            c.execute("UPDATE schedule SET status='Done' WHERE id=?", (plan[0],))
                            conn.commit()
                            st.rerun()

                # 🗑 DELETE BUTTON
                with col3:
                    if st.button("🗑", key=f"delete_{plan[0]}", use_container_width=True):
                        c.execute("DELETE FROM schedule WHERE id=?", (plan[0],))
                        conn.commit()
                        st.rerun()

        else:
            st.info("No study plan added yet 📚")

        st.markdown('</div>', unsafe_allow_html=True)

    # ===== YOUTUBE =====
    st.markdown("""
    <div class="section-header">
        🎥 YouTube
    </div>
    """, unsafe_allow_html=True)
    st.caption("Watch related videos for this topic")
    if st.button("🎥 Learn from YouTube"):
        links = get_youtube_videos(selected_topic)
        for link in links:
            st.markdown(f"[Watch]({link})")

    # ✅ ADD DASHBOARD HERE
    st.markdown('<div class="clean-card">', unsafe_allow_html=True)

    st.markdown("""
    <div class="section-header">
        📊 Dashboard Overview
    </div>
    """, unsafe_allow_html=True)

    total_sessions = c.execute("SELECT COUNT(*) FROM history").fetchone()[0]
    total_topics = len(st.session_state.completed_topics)

    col1, col2 = st.columns(2)

    with col1:
        st.markdown(f"""
        <div style="background:#f8fafc; padding:15px; border-radius:10px;">
            <div style="font-size:14px; color:#64748b;">Total Sessions</div>
            <div style="font-size:28px; font-weight:700;">{total_sessions}</div>
        </div>
        """, unsafe_allow_html=True)

    with col2:
        st.markdown(f"""
        <div style="background:#f8fafc; padding:15px; border-radius:10px;">
            <div style="font-size:14px; color:#64748b;">Topics Covered</div>
            <div style="font-size:28px; font-weight:700;">{total_topics}</div>
        </div>
        """, unsafe_allow_html=True)

    import pandas as pd

    st.markdown('<div class="clean-card">', unsafe_allow_html=True)

    st.subheader("📚 Topic Progress")

    df_table = pd.DataFrame({
          "Topic": topics,
          "Completed": [topic in st.session_state.completed_topics for topic in topics]
    })

    edited_df = st.data_editor(df_table, use_container_width=True)

    new_completed = set(
          edited_df[edited_df["Completed"] == True]["Topic"]
    )

    if new_completed != st.session_state.completed_topics:
          st.session_state.completed_topics = new_completed
          st.rerun()

      # ✅ close card OUTSIDE
    st.markdown('</div>', unsafe_allow_html=True)


    # ===== ANALYTICS  =====

    st.markdown("""
    <div class="section-header">
        📅 Study Analytics
    </div>
    """, unsafe_allow_html=True)
    st.markdown('<div class="result-box">', unsafe_allow_html=True)

    # 🔘 Toggle
    view = st.radio(
        "Select View",
        ["Daily", "Monthly"],
        horizontal=True
    )

    # 📊 FETCH DATA
    data = c.execute("SELECT type, topic, date FROM history").fetchall()

    if data:

        df = pd.DataFrame(data, columns=["type", "topic", "date"])
        df["date"] = pd.to_datetime(df["date"])

        # 🔢 METRICS
        total_sessions = len(df)
        total_topics = df["topic"].nunique()

        unique_days = sorted(set(df["date"].dt.date))
        streak = 1
        for i in range(1, len(unique_days)):
            if (unique_days[i] - unique_days[i-1]).days == 1:
                streak += 1

        # 📊 TOP CARDS
        colA, colB, colC = st.columns(3)
        colA.metric("📘 Sessions", total_sessions)
        colB.metric("📚 Topics", total_topics)
        colC.metric("🔥 Streak", f"{streak} days")

        # 📈 DATA GROUPING
        if view == "Daily":
            df["day"] = df["date"].dt.date
            grouped = df.groupby("day").size()

            x = grouped.index
            y = grouped.values

        else:
            current_month = datetime.now().month
            current_year = datetime.now().year

            df = df[
                (df["date"].dt.month == current_month) &
                (df["date"].dt.year == current_year)
            ]

            df["day"] = df["date"].dt.day
            grouped = df.groupby("day").size()

            days_in_month = 31
            x = list(range(1, days_in_month + 1))
            y = [grouped.get(day, 0) for day in x]

        # 📊 CREATE GRAPH
        fig = go.Figure()

        # 🔵 CLEAN BAR (no fat rectangle)
        fig.add_trace(go.Bar(
            x=x,
            y=y,
            name="Activity",
            marker=dict(
                color="#22d3ee"  # cyan
            ),
            width=0.4,   # 👈 IMPORTANT (fix big block issue)
            opacity=0.85
        ))

        # ⚪ LINE (VISIBLE, NOT WHITE ON WHITE)
        cumulative = pd.Series(y).cumsum()

        fig.add_trace(go.Scatter(
            x=x,
            y=y,   # 🔥 use actual values (not cumulative)
            mode="lines+markers",
            line=dict(color="#6366f1", width=3),
            marker=dict(size=6, color="#6366f1"),
            name="Activity"
        ))

        # 🎨 FIXED DARK THEME (HIGH CONTRAST)
        fig.update_layout(
            height=380,
            paper_bgcolor="rgba(0,0,0,0)",   # transparent
            plot_bgcolor="rgba(0,0,0,0)",

            margin=dict(l=10, r=10, t=20, b=10),

            xaxis=dict(
                showgrid=False,
                color="#cbd5f5"
            ),

            yaxis=dict(
                showgrid=True,
                gridcolor="rgba(255,255,255,0.2)",
            ),

            legend=dict(
                orientation="h",
                y=1.05,
                x=1,
                xanchor="right"
            )
        )

        # 📊 LAYOUT SPLIT
        col1, col2 = st.columns([3, 1])

        with col1:
            st.plotly_chart(fig, use_container_width=True)

        with col2:
            st.markdown("### 📌 Activity Breakdown")

            type_counts = df["type"].value_counts()

            for t, count in type_counts.items():
                st.markdown(f"""
                <div style="
                    background:#f1f5f9;
                    padding:10px;
                    border-radius:8px;
                    margin-bottom:8px;
                    font-weight:500;
                ">
                    {t}: <b>{count}</b>
                </div>
                """, unsafe_allow_html=True)
                st.markdown('</div>', unsafe_allow_html=True)

    else:
      st.info("No study data yet 📚")



In [ ]:
from pyngrok import ngrok
ngrok.kill()

In [ ]:
import subprocess, time
process = subprocess.Popen(["streamlit", "run", "app.py"])
time.sleep(5)

In [ ]:
from pyngrok import ngrok

ngrok.set_auth_token("3AsXDHdlpj8GJGHeEysjPOTTj0R_6WPAXSDiNCyZSZNXm6cWW")

public_url = ngrok.connect(8501)
print("OPEN THIS LINK:", public_url)